# Inspect KodCode Code-R1 Data Structure

This notebook inspects the local KodCode Code-R1 JSONL files used for SFT, GRPO, and eval.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd

PROJECT = Path.cwd()
DATA = PROJECT / 'data'

FILES = {
    'sft': DATA / 'kodcode_code_r1_sft.jsonl',
    'sft_prompts': DATA / 'kodcode_code_r1_sft_prompts.jsonl',
    'grpo': DATA / 'kodcode_code_r1_grpo_631.jsonl',
    'eval_631': DATA / 'kodcode_code_r1_eval_631.jsonl',
    'eval_532': DATA / 'kodcode_code_r1_eval_532.jsonl',
    'raw_flat': DATA / 'kodcode_light_rl_10k_raw.jsonl',
}

def load_jsonl(path, limit=None):
    rows = []
    with Path(path).open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def maybe_json(value):
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value

for name, path in FILES.items():
    print(f'{name:12s}', path.exists(), path)

## Row Counts and Top-Level Keys

In [ ]:
summary = []
for name, path in FILES.items():
    if not path.exists():
        continue
    rows = load_jsonl(path)
    keys = sorted(rows[0].keys()) if rows else []
    summary.append({'name': name, 'rows': len(rows), 'keys': keys, 'path': str(path)})

pd.DataFrame(summary)

## Inspect Raw KodCode Rows

`raw_flat` is the flattened original KodCode-Light-RL-10K file. It is the source used by `prepare-kodcode-code-r1-data.py` before converting rows into Code-R1 `prompt` / `label` / `metadata` format.

In [ ]:
raw_rows = load_jsonl(FILES['raw_flat'], limit=1000)
raw = raw_rows[0]

print('top-level keys:')
for k in raw.keys():
    print('-', k, type(raw[k]).__name__)

print('\nraw row preview:')
for k, v in raw.items():
    text = json.dumps(v, ensure_ascii=False, indent=2) if not isinstance(v, str) else v
    print('\n' + '=' * 100)
    print(k)
    print('-' * 100)
    print(text[:3000])

In [ ]:
raw_all = load_jsonl(FILES['raw_flat'])
raw_df = pd.DataFrame([
    {
        'question_id': r.get('question_id'),
        'split': r.get('_split'),
        'style': r.get('style'),
        'subset': r.get('subset'),
        'gpt_difficulty': r.get('gpt_difficulty'),
        'gpt_pass_percentage': r.get('gpt_pass_percentage'),
        'r1_correctness': r.get('r1_correctness'),
        'has_solution': bool(str(r.get('solution') or '').strip()),
        'has_r1_solution': bool(str(r.get('r1_solution') or '').strip()),
        'has_test': bool(str(r.get('test') or '').strip()),
        'num_test_info': len(r.get('test_info') or []),
    }
    for r in raw_all
])

display(raw_df.head())
display(raw_df[['split', 'style', 'subset', 'gpt_difficulty', 'r1_correctness']].describe(include='all'))
display(raw_df['gpt_difficulty'].value_counts(dropna=False).rename('count').to_frame())
display(raw_df['subset'].value_counts(dropna=False).head(20).rename('count').to_frame())

## Raw to Code-R1 Field Mapping

The conversion happens in `prepare-kodcode-code-r1-data.py`. Important mapping:

| Raw field | Code-R1 usage |
|---|---|
| `question` | Goes into the user prompt as the problem statement |
| `test_info[0].function_name` | Determines the callable function name |
| `test_info[0].function_declaration` | Added to the prompt as the required signature |
| `test` | Parsed into public tests and hidden tests |
| first N parsed tests | Stored in `metadata.public_tests`; visible to the model |
| remaining parsed tests | Stored in `label`; used as hidden reward/eval tests |
| `solution` / `r1_solution` | Used as canonical solution for SFT trajectory generation |
| `gpt_difficulty`, `subset`, `style`, `question_id` | Stored in `metadata` for filtering and analysis |


## Inspect One GRPO/Eval Prompt Row

GRPO and eval rows usually have `prompt`, `label`, and `metadata`.

In [ ]:
rows = load_jsonl(FILES['grpo'])
row = rows[0]

print('top-level keys:', row.keys())
print('metadata keys:', sorted(row.get('metadata', {}).keys()))
print('\nmetadata:')
print(json.dumps(row.get('metadata', {}), ensure_ascii=False, indent=2)[:4000])
print('\nlabel / hidden tests:')
print(json.dumps(maybe_json(row.get('label')), ensure_ascii=False, indent=2)[:4000])

In [ ]:
prompt = row['prompt']
for i, msg in enumerate(prompt):
    print('=' * 100)
    print(i, msg.get('role'))
    print('-' * 100)
    print(msg.get('content', '')[:5000])

## Inspect One SFT Trajectory

SFT rows usually have `messages`; assistant messages include `<test>` and/or `<answer>` actions.

In [ ]:
sft_rows = load_jsonl(FILES['sft'])
sft = sft_rows[0]
print('top-level keys:', sft.keys())
print('num messages:', len(sft['messages']))

for i, msg in enumerate(sft['messages']):
    print('=' * 100)
    print(i, msg.get('role'))
    print('-' * 100)
    print(msg.get('content', '')[:3000])

## Dataset Composition

In [ ]:
def metadata_table(path):
    rows = load_jsonl(path)
    out = []
    for r in rows:
        md = r.get('metadata') or {}
        out.append({
            'id': md.get('id'),
            'difficulty': md.get('difficulty'),
            'subset': md.get('subset'),
            'style': md.get('style'),
            'protocol': md.get('execution_protocol'),
            'fn_name': md.get('fn_name'),
            'num_public_tests': md.get('num_public_tests'),
            'num_hidden_tests': md.get('num_hidden_tests'),
        })
    return pd.DataFrame(out)

for name in ['sft_prompts', 'grpo', 'eval_631', 'eval_532']:
    path = FILES[name]
    if not path.exists():
        continue
    df = metadata_table(path)
    print('\n', '=' * 100)
    print(name, 'rows=', len(df))
    display(df[['difficulty', 'subset', 'protocol']].describe(include='all'))
    display(df['difficulty'].value_counts(dropna=False).rename('count').to_frame())
    display(df.head())

## Split Overlap Check

This checks whether SFT prompt, GRPO, and eval splits share question ids.

In [ ]:
id_sets = {}
for name in ['sft_prompts', 'grpo', 'eval_631', 'eval_532']:
    if FILES[name].exists():
        df = metadata_table(FILES[name])
        id_sets[name] = set(df['id'].dropna())

names = list(id_sets)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        print(f'{a} vs {b}: overlap={len(id_sets[a] & id_sets[b])}')

## Search by Function, Difficulty, or Subset

In [ ]:
df = metadata_table(FILES['grpo'])

# Change these filters as needed.
difficulty = None  # e.g. 'hard', 'medium', 'easy'
fn_contains = None  # e.g. 'sort'
subset_contains = None

view = df.copy()
if difficulty:
    view = view[view['difficulty'] == difficulty]
if fn_contains:
    view = view[view['fn_name'].fillna('').str.contains(fn_contains, case=False)]
if subset_contains:
    view = view[view['subset'].fillna('').str.contains(subset_contains, case=False)]

display(view.head(20))